In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


def process_file(file_name):

    print(f"Starting processing: {file_name}")

    landing_path = "/Volumes/banking_mini/bronze/landing/"
    file_path = landing_path + file_name

    # --------------------------------------------------
    # 1. Check whether file was already successfully processed
    # --------------------------------------------------

    control_df = spark.table(
        "banking_mini.control.file_processing"
    )

    already_processed = (
        control_df
        .filter(
            (F.col("file_name") == file_name) &
            (F.col("status") == "SUCCESS")
        )
        .limit(1)
        .count() > 0
    )

    if already_processed:
        print(f"SKIPPED: {file_name} was already processed.")
        return

    # --------------------------------------------------
    # 2. Read source file
    # --------------------------------------------------

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    records_received = df.count()

    print(f"Records received: {records_received}")

    # --------------------------------------------------
    # 3. Add Bronze metadata
    # --------------------------------------------------

    bronze_df = (
        df
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.lit(file_name))
    )

    # --------------------------------------------------
    # 4. Write Bronze
    # --------------------------------------------------

    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("banking_mini.bronze.transactions")

    # --------------------------------------------------
    # 5. Data Quality validation
    # --------------------------------------------------

    invalid_df = bronze_df.filter(
        F.col("transaction_id").isNull()
        | F.col("customer_id").isNull()
        | (F.col("amount") <= 0)
        | (~F.col("payment_type").isin(
            "UPI",
            "CARD",
            "NET_BANKING"
        ))
    )

    records_rejected = invalid_df.count()

    # --------------------------------------------------
    # 6. Quarantine invalid records
    # --------------------------------------------------

    quarantine_df = (
        invalid_df
        .withColumn(
            "rejection_reason",
            F.when(
                F.col("transaction_id").isNull(),
                "NULL_TRANSACTION_ID"
            )
            .when(
                F.col("customer_id").isNull(),
                "NULL_CUSTOMER_ID"
            )
            .when(
                F.col("amount") <= 0,
                "INVALID_AMOUNT"
            )
            .when(
                ~F.col("payment_type").isin(
                    "UPI",
                    "CARD",
                    "NET_BANKING"
                ),
                "INVALID_PAYMENT_TYPE"
            )
        )
        .withColumn(
            "rejection_timestamp",
            F.current_timestamp()
        )
    )

    if records_rejected > 0:

        quarantine_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(
                "banking_mini.quarantine.transactions"
            )

    # --------------------------------------------------
    # 7. Keep valid records
    # --------------------------------------------------

    valid_df = bronze_df.filter(
        F.col("transaction_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & (F.col("amount") > 0)
        & F.col("payment_type").isin(
            "UPI",
            "CARD",
            "NET_BANKING"
        )
    )

    # --------------------------------------------------
    # 8. Deduplicate within the incoming file
    # --------------------------------------------------

    silver_df = valid_df.dropDuplicates(
        ["transaction_id"]
    )

    records_processed = silver_df.count()

    # --------------------------------------------------
    # 9. Idempotent MERGE into Silver
    # --------------------------------------------------

    silver_table = DeltaTable.forName(
        spark,
        "banking_mini.silver.transactions"
    )

    (
        silver_table.alias("target")
        .merge(
            silver_df.alias("source"),
            "target.transaction_id = source.transaction_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # --------------------------------------------------
    # 10. Record successful processing
    # --------------------------------------------------

    control_record = spark.createDataFrame(
        [(
            file_name,
            "SUCCESS",
            records_received,
            records_processed,
            records_rejected
        )],
        [
            "file_name",
            "status",
            "records_received",
            "records_processed",
            "records_rejected"
        ]
    ).withColumn(
        "processed_at",
        F.current_timestamp()
    ).withColumn(
        "error_message",
        F.lit(None).cast("string")
    )

    control_record.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "banking_mini.control.file_processing"
        )

    print("Processing completed successfully.")
    print(f"Received : {records_received}")
    print(f"Processed: {records_processed}")
    print(f"Rejected : {records_rejected}")

In [0]:
dbutils.widgets.text("file_name", "")

file_name = dbutils.widgets.get("file_name")

if not file_name:
    raise ValueError("file_name parameter is required")

process_file(file_name)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-5062961517679481>, line 6
      3 file_name = dbutils.widgets.get("file_name")
      5 if not file_name:
----> 6     raise ValueError("file_name parameter is required")
      8 process_file(file_name)

ValueError: file_name parameter is required

In [0]:
%sql
SELECT
    file_name,
    status,
    records_received,
    records_processed,
    records_rejected,
    processed_at
FROM banking_mini.control.file_processing
ORDER BY processed_at;

file_name,status,records_received,records_processed,records_rejected,processed_at
transactions_2026_09_26.csv,SUCCESS,8,6,1,2026-09-26T17:14:05.959Z
transactions_2026_09_27.csv,SUCCESS,6,4,1,2026-09-26T17:39:57.913Z
transactions_2026_09_28.csv,SUCCESS,5,3,1,2026-09-26T17:53:56.758Z


In [0]:
%sql 
SELECT * FROM banking_mini.silver.transactions;

transaction_id,customer_id,amount,transaction_date,payment_type,ingestion_timestamp,source_file
T001,C101,500.0,2026-09-26,UPI,2026-09-26T16:26:42.595Z,dbfs:/Volumes/banking_mini/bronze/landing/transactions_2026_09_26.csv
T002,C102,1200.0,2026-09-26,CARD,2026-09-26T16:26:42.595Z,dbfs:/Volumes/banking_mini/bronze/landing/transactions_2026_09_26.csv
T003,C103,750.0,2026-09-26,NET_BANKING,2026-09-26T16:26:42.595Z,dbfs:/Volumes/banking_mini/bronze/landing/transactions_2026_09_26.csv
T004,C104,250.0,2026-09-26,UPI,2026-09-26T16:26:42.595Z,dbfs:/Volumes/banking_mini/bronze/landing/transactions_2026_09_26.csv
T005,C105,900.0,2026-09-26,CARD,2026-09-26T16:26:42.595Z,dbfs:/Volumes/banking_mini/bronze/landing/transactions_2026_09_26.csv
T007,C107,650.0,2026-09-26,CARD,2026-09-26T16:26:42.595Z,dbfs:/Volumes/banking_mini/bronze/landing/transactions_2026_09_26.csv
T014,C114,900.0,2026-09-28,CARD,2026-09-26T17:53:51.690Z,transactions_2026_09_28.csv
T013,C113,300.0,2026-09-28,UPI,2026-09-26T17:53:51.690Z,transactions_2026_09_28.csv
T016,C116,1200.0,2026-09-28,NET_BANKING,2026-09-26T17:53:51.690Z,transactions_2026_09_28.csv
T011,C111,1500.0,2026-09-27,NET_BANKING,2026-09-26T17:39:54.710Z,transactions_2026_09_27.csv
